# 03. 코퍼스 고정과 Before 실행 준비
목표는 개선 전 서비스를 같은 재료로 다시 실행할 준비를 하는 것입니다. 기존 운영 DB가 없으므로 **재구성한 Before**입니다. 캐시를 운영 코퍼스와 같다고 주장하지 않습니다.

기본 Run All은 로컬 캐시 조사·스냅샷·모의 검증만 수행합니다. GPU·모델 다운로드·법령 API·원격 DB 호출은 하지 않습니다. 실제 답변 수집은 마지막의 승인 셀에서만 가능하며 기본값은 꺼져 있습니다.

단계 1 최신 작업을 포함한 feat/before-prep에서 작성합니다. 원격 main에 빠진 PR #8/#9 변경은 이 브랜치에 포함되며 원격 병합은 별도입니다. 서비스 노드와 기존 prepare.py/rag.py를 수정하지 않습니다.

질문별 답변 전문·시간·출처와 검토 메모는 [Before 답변 검토 노트북](03_before_review.ipynb)에서 확인하세요.

In [1]:
import json
import os
import sys
from pathlib import Path
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
NOTEBOOK = ROOT / "notebooks/redesign/03_metadata_and_ingest.ipynb"
CACHE = Path(os.environ.get("AIGO_CACHE_DIR", Path.home() / ".cache/aigo-ai"))
SOURCE = CACHE / "corpus_v1-seed.jsonl"
MANIFEST = ROOT / "notebooks/redesign/data/before_manifest.json"

## 1. 원문은 보존하고, 식별자와 누락을 기록합니다
기존 key/version_key만으로 문서를 합치지 않습니다. 동일 사건번호도 서로 다른 원문일 수 있으므로 출처를 함께 사용합니다. 동일 식별자의 서로 다른 본문은 본문 해시로 구별하고 보고합니다. 원문 행은 삭제하지 않습니다.

판례의 선고일은 법령 시행일이 아닙니다. 날짜 종류를 분리하고 없는 날짜는 null입니다. 현행 별칭 URL은 특정 과거 판본을 보장하지 않으므로 별도로 셉니다. URL 검사는 형식 검사이며 원문 내용·최신성 검증이 아닙니다.

문서 형식 변환은 준비용입니다. 법령은 조문 그대로, 판례·해석례는 캐시의 필드 순서로 합칩니다. 새 청킹 실험은 하지 않으며 긴 문서의 임베딩 잘림 가능성은 실제 환경 확인 때 검토합니다.

In [2]:
import asyncio
import hashlib
import re
import time
from collections import Counter
from datetime import date
from pathlib import Path
from urllib.parse import parse_qs, unquote, urlsplit


def sha256(data):
    """bytes를 받아 재현 확인용 SHA-256 문자열을 반환합니다."""
    return hashlib.sha256(data).hexdigest()


def json_bytes(value):
    """JSON 값을 키 순서와 공백이 고정된 UTF-8 bytes로 반환합니다."""
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":")).encode("utf-8")


def valid_source(url):
    """캐시 URL이 인증정보 없는 국가법령정보 원문 주소 형식인지 반환합니다."""
    parts = urlsplit(url or "")
    if parts.scheme != "https" or parts.netloc not in {"law.go.kr", "www.law.go.kr"}:
        return False
    if parts.fragment or parts.username or parts.password:
        return False
    query = parse_qs(parts.query, keep_blank_values=True)
    if unquote(parts.path).startswith("/법령/"):
        return not query
    names = {"/LSW/precInfoP.do": "precSeq", "/LSW/expcInfoP.do": "expcSeq"}
    key = names.get(parts.path)
    return bool(key and set(query) == {key} and len(query[key]) == 1 and query[key][0].isdigit())


def normalize_document(row):
    """캐시 한 행을 본문·판본·날짜·출처가 분리된 준비 문서로 반환합니다."""
    kind = row["doc_type"]
    if kind not in {"law", "prec", "expc"}:
        raise ValueError("unsupported_document_type")
    text = row.get("text") or "\n\n".join(f"{k}\n{v}" for k, v in row.get("fields", {}).items() if v)
    if not text.strip():
        raise ValueError("empty_document")
    if not valid_source(row.get("url")):
        raise ValueError("invalid_source_url")
    for key in ("valid_from", "valid_to"):
        if row.get(key):
            date.fromisoformat(row[key])
    if row.get("valid_from") and row.get("valid_to") and row["valid_from"] > row["valid_to"]:
        raise ValueError("reversed_validity")
    version = row.get("version_key") or None
    identity = [kind, row.get("key"), version, row["url"]]
    return {
        "document_id": sha256(json_bytes(identity)), "body_sha256": sha256(text.encode("utf-8")),
        "doc_type": kind, "source_key": row.get("key"), "version": version,
        "title": row.get("title"), "article": row.get("article"), "law_name": row.get("law_name"),
        "law_source_id": row.get("law_source_id"),
        "effective_from": row.get("valid_from") if kind == "law" else None,
        "effective_to": row.get("valid_to") if kind == "law" else None,
        "decision_date": row.get("valid_from") if kind != "law" else None,
        "url": row["url"], "text": text,
        "date_provenance": "inherited_cache_not_reverified",
    }


def prepare_corpus(rows):
    """캐시 행 목록에서 결정론적 문서 목록과 누락·중복 통계를 반환합니다."""
    docs = [normalize_document(row) for row in rows]
    docs.sort(key=lambda d: (d["document_id"], d["body_sha256"]))
    pairs = Counter((d["document_id"], d["body_sha256"]) for d in docs)
    ids = Counter(d["document_id"] for d in docs)
    return docs, {
        "documents": len(docs), "types": dict(sorted(Counter(d["doc_type"] for d in docs).items())),
        "duplicate_source_version_keys": sum(n - 1 for n in Counter(r.get("version_key") for r in rows).values() if n > 1),
        "exact_duplicate_rows": sum(n - 1 for n in pairs.values() if n > 1),
        "identity_conflicts": sum(len({d["body_sha256"] for d in docs if d["document_id"] == key}) > 1 for key, n in ids.items() if n > 1),
        "missing_version": sum(d["version"] is None for d in docs),
        "missing_applicable_date": sum(not (d["effective_from"] or d["decision_date"]) for d in docs),
        "law_alias_urls": sum(d["doc_type"] == "law" and "/(" not in unquote(d["url"]) for d in docs),
    }


def write_once(path, content):
    """동일한 파일은 재사용하고 다른 내용은 덮어쓰지 않습니다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        if path.read_bytes() != content:
            raise ValueError("snapshot_content_changed")
    else:
        with path.open("xb") as handle:
            handle.write(content)


def service_payload(doc):
    """준비 문서를 기존 서비스의 한국어 payload 필드로 변환합니다."""
    kind = doc["doc_type"]
    query = parse_qs(urlsplit(doc["url"]).query)
    source_id = (query.get("precSeq") or query.get("expcSeq") or [None])[0]
    # 법령ID는 기존 검색 API 캐시에서 확인한 값만 사용합니다.
    if kind == "law":
        source_id = doc.get("law_source_id")
        if not source_id:
            raise ValueError("law_source_id_required_for_ingest")
    return {
        "doc_type": {"law": "법령", "prec": "판례", "expc": "법령해석례"}[kind],
        "title": doc["law_name"] if kind == "law" else doc["title"],
        "source_id": source_id, "chunk_id": doc["source_key"].removeprefix("해석례|") if kind == "expc" else doc["document_id"], "text": doc["text"],
        "조문번호": doc["article"], "사건번호": doc["source_key"].removeprefix("판례|") if kind == "prec" else None,
        "document_id": doc["document_id"], "version": doc["version"], "body_sha256": doc["body_sha256"],
        "effective_from": doc["effective_from"], "effective_to": doc["effective_to"],
        "decision_date": doc["decision_date"], "source_url": doc["url"],
    }

In [3]:
# 동일 사건번호라도 출처가 다르면 별개의 문서로 유지하는 작은 예입니다.
example = {"doc_type": "prec", "key": "판례|합성사건", "version_key": "판례|합성사건",
           "title": "합성 판례", "text": "테스트 본문", "url": "https://www.law.go.kr/LSW/precInfoP.do?precSeq=1"}
a = normalize_document(example)
b = normalize_document({**example, "url": "https://www.law.go.kr/LSW/precInfoP.do?precSeq=2"})
assert a["document_id"] != b["document_id"]
assert a["decision_date"] is None
print("합성 예: 출처가 다른 문서를 보존하고 없는 날짜는 null로 유지")

합성 예: 출처가 다른 문서를 보존하고 없는 날짜는 null로 유지


## 2. 캐시 조사와 변경 불가능한 로컬 사본
원본의 SHA-1을 과거 manifest와 대조한 뒤 SHA-256으로 새 사본을 고정합니다. 재실행해도 같은 사본을 재사용하며 기존 파일을 덮어쓰지 않습니다. source.jsonl은 원본 bytes 그대로, corpus.jsonl은 준비 형식입니다. 둘 다 저장소 밖 캐시에 보관합니다.

복구는 이 사본을 다른 환경에 복사하고 아래 manifest의 해시와 대조하는 방식입니다. 원본 API를 다시 조회하면 최신 내용으로 바뀔 수 있으므로 동일 자료 복구로 간주하지 않습니다. 사본 자체를 잃으면 해시만으로 내용을 복원할 수 없습니다.

본문 캐시에 법령ID가 없어 그대로 서비스에 넣으면 인용 중복 처리가 왜곡될 수 있습니다. 기존 검색 API 캐시 8개에서 실제 법령ID를 복구하고 조회 파일 해시와 매핑을 함께 보관합니다. 새 API 조회나 가짜 ID 생성은 하지 않습니다.


깨끗한 환경에서 재실행하려면 스냅샷 전체를 별도로 백업하세요. source.jsonl을 새 AIGO_CACHE_DIR/corpus_v1-seed.jsonl로, source_manifest.json을 manifest_v1-seed.json로, api/의 8개 조회 파일을 같은 캐시의 api/로 복원합니다. corpus.jsonl과 law_ids.json도 함께 보존합니다. API 재조회 없이 같은 노트북으로 해시를 대조할 수 있습니다.


In [4]:
source_bytes = SOURCE.read_bytes()
old_manifest = json.loads((CACHE / "manifest_v1-seed.json").read_text(encoding="utf-8"))
assert hashlib.sha1(source_bytes).hexdigest() == old_manifest["sha1"]
rows = [json.loads(line) for line in source_bytes.decode("utf-8").splitlines() if line.strip()]
# 본문 캐시에 빠진 법령ID만 과거 검색 캐시에서 복구합니다. 네트워크 요청은 없습니다.
law_ids = {}
lookup_hashes = {}
lookup_files = {}
for spec in old_manifest["sources"]["laws"]:
    params = {"target": "eflaw" if spec["history"] else "law", "query": spec["name"], "display": 100}
    cache_key = hashlib.sha1(json.dumps(["https://www.law.go.kr/DRF/lawSearch.do", params], sort_keys=True, ensure_ascii=False).encode()).hexdigest()
    lookup_bytes = (CACHE / "api" / (cache_key + ".json")).read_bytes()
    hits = json.loads(lookup_bytes)["LawSearch"]["law"]
    hits = hits if isinstance(hits, list) else [hits]
    ids = {row["법령ID"] for row in hits if row["법령명한글"] == spec["name"]}
    assert len(ids) == 1, "법령 ID를 확정할 수 없습니다"
    law_ids[spec["name"]] = ids.pop()
    lookup_hashes[cache_key + ".json"] = sha256(lookup_bytes)
    lookup_files[cache_key + ".json"] = lookup_bytes
rows_with_ids = [{**row, "law_source_id": law_ids[row["law_name"]]} if row["doc_type"] == "law" else row for row in rows]
documents, audit = prepare_corpus(rows_with_ids)
assert len(rows) == old_manifest["n_docs"]
prepared_bytes = b"".join(json_bytes(doc) + b"\n" for doc in documents)
CORPUS_SHA = sha256(prepared_bytes)
SNAPSHOT = CACHE / "before" / CORPUS_SHA
assert not SNAPSHOT.resolve().is_relative_to(ROOT.resolve())
write_once(SNAPSHOT / "source.jsonl", source_bytes)
write_once(SNAPSHOT / "source_manifest.json", (CACHE / "manifest_v1-seed.json").read_bytes())
for name, content in lookup_files.items():
    write_once(SNAPSHOT / "api" / name, content)
write_once(SNAPSHOT / "corpus.jsonl", prepared_bytes)
write_once(SNAPSHOT / "law_ids.json", json_bytes({"ids": law_ids, "lookup_sha256": lookup_hashes}))
assert SOURCE.read_bytes() == source_bytes
assert prepare_corpus(list(reversed(rows_with_ids)))[0] == documents
print(json.dumps(audit, ensure_ascii=False, indent=2))

{
  "documents": 2494,
  "types": {
    "expc": 55,
    "law": 2097,
    "prec": 342
  },
  "duplicate_source_version_keys": 1,
  "exact_duplicate_rows": 0,
  "identity_conflicts": 0,
  "missing_version": 0,
  "missing_applicable_date": 55,
  "law_alias_urls": 1711
}


## 3. 코드·설정·프롬프트 고정
기준 app 코드는 08d686b입니다. 파일 해시로 현재 app이 같은지 확인합니다. 임베딩/생성 모델은 **기존 기본 설정을 명시적으로 선택**합니다. .env를 읽어 비밀을 복사하지 않으며 운영 설정이라고 주장하지 않습니다.

temperature=0.3, timeout=90초, max_retries=0은 기존 LLM 클라이언트 값입니다. CONTEXT_TOP_K=3은 설정에 있지만 현재 생성기는 retrieved_docs 전체를 사용합니다. 설정값과 실제 전달 문맥을 구분합니다.

모델 가중치 revision·양자화·서버·GPU는 아직 미정입니다. 이 항목이 확정되고 비용 승인이 나기 전에는 실제 실행을 막습니다. 실행 환경 확정 시 별도의 실제 실행 기록에 유효 설정과 패키지 버전을 남깁니다.

In [5]:
import ast
import importlib.metadata
import subprocess

def git_read(*args):
    """현재 저장소의 Git 읽기 명령 결과를 bytes로 반환합니다."""
    return subprocess.check_output(["git", "-c", f"safe.directory={ROOT.as_posix()}", *args], cwd=ROOT)

BEFORE_COMMIT = "08d686b31904b90e9aa5d84e8803e03f51567cc2"
app_files = git_read("ls-files", "app").decode().splitlines()
source_hashes = {name: sha256((ROOT / name).read_bytes()) for name in app_files}
assert all((ROOT / name).read_bytes().replace(b"\r\n", b"\n") == git_read("show", f"{BEFORE_COMMIT}:{name}").replace(b"\r\n", b"\n") for name in app_files)
# 프롬프트는 AST로 문자열만 읽으므로 서비스 import나 LLM 초기화가 없습니다.
prompt_hashes = {}
for file, name in [("app/graph/nodes/generator.py", "GENERATOR_SYSTEM_PROMPT"),
                   ("app/graph/nodes/understand_query.py", "UNDERSTAND_QUERY_SYSTEM_PROMPT")]:
    tree = ast.parse((ROOT / file).read_text(encoding="utf-8"))
    value = next(ast.literal_eval(node.value) for node in tree.body if isinstance(node, ast.Assign)
                 and any(isinstance(t, ast.Name) and t.id == name for t in node.targets))
    prompt_hashes[name] = sha256(value.encode("utf-8"))
EVAL_PATH = ROOT / "notebooks/redesign/eval/eval_set.json"
eval_data = json.loads(EVAL_PATH.read_text(encoding="utf-8"))
questions = [{"id": item["id"], "question": item["question"]} for item in eval_data["items"]]
assert len(questions) == 14
BASELINE_SETTINGS = {"EMBEDDING_MODEL": "nlpai-lab/KURE-v1",
                     "LLM_MODEL": "LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct",
                     "RETRIEVAL_TOP_K": 5, "CONTEXT_TOP_K": 3, "RELEVANCE_THRESHOLD": 0.35}
manifest = {
    "label": "reconstructed_before", "status": "local_preparation_only", "prepared_on": "2026-10-17",
    "before_commit": BEFORE_COMMIT, "preparation_base": "53fbc3f8401668a899c33f93d52baab981177d18",
    "source_sha256": sha256(source_bytes), "corpus_sha256": CORPUS_SHA,
    "law_id_lookup_sha256": lookup_hashes,
    "source_cache_version": old_manifest["corpus_version"], "source_built": old_manifest["built"],
    "audit": audit, "app_files_sha256": sha256(json_bytes(source_hashes)),
    "prompt_sha256": prompt_hashes, "eval_version": eval_data["version"],
    "eval_sha256": sha256(EVAL_PATH.read_bytes()), "question_count": len(questions),
    "settings": BASELINE_SETTINGS, "llm_client": {"temperature": 0.3, "timeout": 90, "max_retries": 0},
    "pyproject_sha256": sha256((ROOT / "pyproject.toml").read_bytes()),
    "lock_sha256": sha256((ROOT / "uv.lock").read_bytes()),
    "local_packages": {name: importlib.metadata.version(name) for name in
                       ["langgraph", "langchain-core", "langchain-openai", "qdrant-client", "sentence-transformers", "torch"]},
    "pending_runtime": ["generation_revision", "embedding_revision", "quantization", "GPU", "server", "cost_and_time_approval"],
    "restore": "AIGO_CACHE_DIR/before/<corpus_sha256>의 source.jsonl·corpus.jsonl·law_ids.json 사본을 별도 백업에서 복사하고 SHA-256 대조. 기본 캐시는 사용자 홈/.cache/aigo-ai. 현재 동일 로컬 디스크 사본만 만들었으며 외부 백업은 미완료.",
    "limitations": ["운영 DB 복원이 아닌 v1-seed 캐시 기반 재구성", "법령 날짜·판본은 캐시에서 상속, 원문 최신성 재검증 아님",
                    "날짜 없는 해석례와 현행 별칭 URL은 시간 유효성 미확정", "실제 모델 호출·색인·성능 점수 없음"],
}
# 준비한 입력은 고정하고 실행 OS/설치 패키지 차이는 runtime에 별도 기록합니다.
execution_packages = manifest["local_packages"]
if MANIFEST.exists():
    frozen_manifest = json.loads(MANIFEST.read_text(encoding="utf-8"))
    assert {k: v for k, v in manifest.items() if k != "local_packages"} == {k: v for k, v in frozen_manifest.items() if k != "local_packages"}, "고정 입력이 변경되었습니다"
    manifest = frozen_manifest
else:
    write_once(MANIFEST, json.dumps(manifest, ensure_ascii=False, indent=2).encode("utf-8") + b"\n")
print("고정 입력 대조 완료. 실제 실행 환경은 runtime에 별도 기록")


고정 입력 대조 완료. 실제 실행 환경은 runtime에 별도 기록


## 4. 현재 서비스의 결과를 그대로 수집합니다
공식 [LangChain usage 설명](https://reference.langchain.com/python/langchain-core/messages/ai/UsageMetadata)과 달리 현재 SSE의 total_tokens는 문자열 길이를 누적합니다. 이를 실제 usage로 쓰지 않고 raw 이벤트만 보존하며 usage·비용은 unknown입니다.

입력은 id와 question만입니다. 정답·기대 행동·근거는 전달하지 않습니다. 실제 호출은 stream_chat(question) 한 번이며, run_preformat과 stream_formatter를 관찰해 검색 상태와 생성기에 전달되는 메시지를 기록합니다. 원래 함수 호출은 한 번만 수행하고 반환값·순서·fallback은 바꾸지 않습니다.

문맥은 실제 formatter 호출 때만 기록합니다. fallback이면 생성 문맥은 null입니다. 서버 스트리밍 메시지와 단계 지연을 구분하며 자동 재시도는 하지 않습니다. 결과는 실행당 새 파일로 남겨 실패도 보존합니다. 모의 결과는 임시 폴더에서만 검증합니다.

In [6]:
async def collect_answer(question, stream, trace):
    """질문과 SSE 함수·관찰 dict에서 답변/인용/실패/지연 기록을 반환합니다."""
    started = time.perf_counter()
    result = {"id": question["id"], "question": question["question"], "status": "failed",
              "answer": "", "citations": [], "events": [], "error": None,
              "ttft_ms": None, "total_ms": None, "usage": "unknown", "cost": "unknown"}
    ended = False
    try:
        async for event in stream(question["question"]):
            elapsed = (time.perf_counter() - started) * 1000
            result["events"].append({"elapsed_ms": elapsed, **event})
            kind, data = event["event"], event["data"]
            if kind == "token" and data.get("delta"):
                if result["ttft_ms"] is None:
                    result["ttft_ms"] = elapsed
                result["answer"] += data["delta"]
            elif kind == "citation":
                result["citations"].append(data)
            elif kind == "error":
                result["error"] = data.get("code", "SERVICE_ERROR")
            elif kind == "message_end":
                ended = True
    except asyncio.CancelledError:
        # 호출자가 취소한 경우 성공으로 저장하지 않으며 배치도 중단시킵니다.
        raise
    except Exception as exc:
        # 예외 본문에 URL 인증키가 섞일 수 있어 타입만 기록합니다.
        result["error"] = type(exc).__name__
    result["total_ms"] = (time.perf_counter() - started) * 1000
    if not result["error"]:
        result["error"] = None if ended and result["answer"].strip() else ("EMPTY_RESPONSE" if ended else "INCOMPLETE_STREAM")
    result["status"] = "ok" if result["error"] is None else "failed"
    result["trace"] = dict(trace)
    return result


def save_trace(path, record):
    """한 질문 결과를 새 JSON 파일로 저장합니다. 기존 결과 덮어쓰기는 금지합니다."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("x", encoding="utf-8") as handle:
        json.dump(record, handle, ensure_ascii=False, indent=2)


async def run_current_service(question):
    """승인된 실행 환경에서 기존 stream_chat을 관찰하여 한 질문 결과를 반환합니다."""
    from unittest.mock import patch
    from app.services import chat_service
    from app.graph.nodes.generator import _build_messages
    trace = {"retrieved_docs": None, "generation_messages": None, "preformat_ms": None}
    original_preformat = chat_service.run_preformat
    original_formatter = chat_service.stream_formatter

    def observe_preformat(query):
        started = time.perf_counter()
        try:
            state = original_preformat(query)
            trace["retrieved_docs"] = state.get("retrieved_docs", [])
            trace["retrieval_passed"] = state.get("retrieval_passed")
            trace["user_query"] = state.get("user_query")
            return state
        finally:
            trace["preformat_ms"] = (time.perf_counter() - started) * 1000

    def observe_formatter(state):
        trace["generation_messages"] = [{"type": m.type, "content": m.content} for m in _build_messages(state)]
        yield from original_formatter(state)

    # 전역 계측 충돌을 피하도록 이 노트북은 질문을 병렬 실행하지 않습니다.
    with patch.object(chat_service, "run_preformat", observe_preformat), patch.object(chat_service, "stream_formatter", observe_formatter):
        return await collect_answer(question, chat_service.stream_chat, trace)

## 5. 네트워크 없는 검증
같은 입력의 ID/해시, 중복·다른 원문, 판본·날짜 누락, 잘못된 URL, 실패·빈 응답·중간 종료를 확인합니다. 테스트에 실제 개인정보를 사용하지 않습니다. 아래 결과는 합성 fixture이며 Before 성능 수치가 아닙니다.

In [7]:
import tempfile

assert normalize_document(example) == normalize_document(dict(reversed(list(example.items()))))
assert normalize_document({**example, "text": "다른 본문"})["body_sha256"] != a["body_sha256"]
assert prepare_corpus([example, example])[1]["exact_duplicate_rows"] == 1
assert prepare_corpus([example, {**example, "text": "변경"}])[1]["identity_conflicts"] == 1
missing = normalize_document({**example, "version_key": None})
assert missing["version"] is None and missing["effective_from"] is None
for bad in [{**example, "text": " "}, {**example, "url": "https://example.com/x"},
            {**example, "url": example["url"] + "&" + "OC" + "=secret"},
            {**example, "valid_from": "not-a-date"}]:
    try:
        normalize_document(bad)
        raise AssertionError("invalid document accepted")
    except ValueError:
        pass
seen = []
async def fake_success(query):
    seen.append(query)
    yield {"event": "token", "data": {"delta": "합성 답변"}}
    yield {"event": "citation", "data": {"url": example["url"]}}
    yield {"event": "message_end", "data": {"total_tokens": 999}}
async def fake_empty(query):
    yield {"event": "message_end", "data": {}}
async def fake_broken(query):
    yield {"event": "token", "data": {"delta": "부분"}}
    raise RuntimeError("합성 실패")
async def fake_incomplete(query):
    yield {"event": "token", "data": {"delta": "부분"}}
fixture = {"id": "fixture", "question": "합성 질문", "gold_sources": ["DO_NOT_SEND"]}
record = await collect_answer(fixture, fake_success, {"retrieved_docs": []})
assert seen == ["합성 질문"] and record["status"] == "ok" and record["usage"] == "unknown"
assert "gold_sources" not in record
assert (await collect_answer(fixture, fake_empty, {}))["error"] == "EMPTY_RESPONSE"
failed = await collect_answer(fixture, fake_broken, {})
assert failed["error"] == "RuntimeError" and failed["answer"] == "부분"
assert (await collect_answer(fixture, fake_incomplete, {}))["error"] == "INCOMPLETE_STREAM"
with tempfile.TemporaryDirectory() as directory:
    path = Path(directory) / "mock.json"
    record["run_kind"] = "fixture_only"
    save_trace(path, record)
    assert json.loads(path.read_text(encoding="utf-8"))["run_kind"] == "fixture_only"
    try:
        save_trace(path, record)
        raise AssertionError("existing trace overwritten")
    except FileExistsError:
        pass
assert SOURCE.read_bytes() == source_bytes
print("PASS: 결정론·중복·누락·출처·실패·빈 응답·저장 검증 (합성 fixture)")
# 실제 모델 대신 서비스 모듈 경계만 대체해 호출 횟수와 문맥 관찰을 검증합니다.
import types
from unittest.mock import patch
calls = []
state_fixture = {"user_input": "합성 질문", "retrieved_docs": [{"text": "합성 근거"}], "retrieval_passed": True}
def mock_preformat(query):
    calls.append(("preformat", query))
    return state_fixture
def mock_formatter(state):
    calls.append(("formatter", state))
    yield "합성 서비스 답변"
service = types.ModuleType("app.services.chat_service")
service.run_preformat = mock_preformat
service.stream_formatter = mock_formatter
async def mock_chat(query):
    state = service.run_preformat(query)
    for chunk in service.stream_formatter(state):
        yield {"event": "token", "data": {"delta": chunk}}
    yield {"event": "message_end", "data": {}}
service.stream_chat = mock_chat
services = types.ModuleType("app.services")
services.chat_service = service
generator = types.ModuleType("app.graph.nodes.generator")
generator._build_messages = lambda state: [types.SimpleNamespace(type="human", content=state["retrieved_docs"][0]["text"])]
with patch.dict(sys.modules, {"app.services": services, "app.graph.nodes.generator": generator}):
    observed = await run_current_service(fixture)
assert [name for name, _ in calls] == ["preformat", "formatter"]
assert observed["trace"]["retrieved_docs"] == state_fixture["retrieved_docs"]
assert observed["trace"]["generation_messages"] == [{"type": "human", "content": "합성 근거"}]
assert not any(name.startswith("app.") for name in sys.modules), "오프라인 실행은 서비스 import 금지"
print("PASS: 서비스 경계 모의 검증 — 입력 질문만 전달, 기존 함수 각 1회 호출, 문맥 관찰")

assert all(service_payload(d)["source_id"] for d in documents)
assert len(law_ids) == 8
print("PASS: 기존 API 캐시의 실제 법령 ID 8개 복구; 전체 서비스 payload 출처 ID 확인")


PASS: 결정론·중복·누락·출처·실패·빈 응답·저장 검증 (합성 fixture)
PASS: 서비스 경계 모의 검증 — 입력 질문만 전달, 기존 함수 각 1회 호출, 문맥 관찰
PASS: 기존 API 캐시의 실제 법령 ID 8개 복구; 전체 서비스 payload 출처 ID 확인


## 6. 승인 후 실행할 셀 — 기본 실행 안 함
RunPod 생성·과금·모델 다운로드는 이 노트북에서 자동으로 시작하지 않습니다. 환경을 확정한 뒤 GPU·revision·양자화·예상 비용·시간 상한을 먼저 승인받습니다. 현재는 실행 준비까지만 완료합니다.

그 다음 새 커널에서 별도 로컬 Qdrant에만 색인하고 EXAONE 호환 엔드포인트로 연결합니다. 기존 DB는 사용하지 않습니다. 아래 실제 실행 셀은 corpus 전체를 한 번 적재하며, 임베딩 길이·모델 revision·서버 한도 확인이 끝나기 전에는 실행하지 않습니다. 긴 문서 잘림 여부와 운영 코퍼스 차이는 결과에 반드시 보고합니다.

[Qdrant point 문서](https://qdrant.tech/documentation/manage-data/points/)에 따라 문서 payload에 안정적인 document_id를 보관합니다. 기존 서비스의 point UUID 생성은 변경하지 않습니다. 법령 ID 단위의 인용 중복 처리와 현행 링크 생성, history 전달 누락, top-1 게이트, SSE 문자 수 계측은 단계 3의 확인 대상입니다.

생성 모델 revision은 승인된 서버 배포 정보와 대조해야 합니다. 클라이언트의 모델명만으로 원격 가중치 동일성을 보장하지 않습니다. 시간 상한은 색인/질문 사이와 응답 대기에서 적용하나 이미 진행 중인 GPU·동기 작업이나 RunPod 과금을 강제 종료하지는 않습니다. 예산 준수에는 승인된 서버의 종료 설정이 별도로 필요합니다. timeout 후에는 해당 실행을 중단하고 새 커널로 재개합니다.


### RunPod 실행 준비 — 2026-10-18
모델 이름만 같아도 파일이 갱신될 수 있어 **판본(commit SHA)**을 고정합니다. 서버도 태그 대신 이미지 digest를 사용합니다. 공개 메타데이터만 조회했으며 이미지·모델 다운로드나 GPU 실행은 하지 않았습니다.

- EXAONE 판본: `553ea250b9a5317231459279d5847d6cf955b9aa`
- KURE 판본: `8b418a58414668e75532ed045c22d9ca018ae2b2`
- vLLM 0.31.0: 해당 버전 registry의 `ExaoneForCausalLM` 지원 확인. GPU에서 실제 시작은 아직 검증하지 않았습니다.
- 로컬 Windows 메모리 부족 확인 후 사용자 승인으로 KURE·Qdrant·수집 노트북도 같은 RunPod의 CPU에서 실행합니다. vLLM과 별도 가상환경을 사용하며 모델과 서비스 로직은 유지합니다. 코퍼스는 비공개 /workspace로 전송하고 해시를 확인합니다. CPU 색인 시간과 실제 Linux 패키지를 기록하며 이전 운영 지연과 같다고 해석하지 않습니다.
- A40 48GB 1장, EXAONE BF16, 4/8bit 양자화 없음. 컨텍스트 32,768·동시 요청 1·GPU 메모리 예약 85%를 명시합니다. 기존 프롬프트·temperature·검색 설정은 유지합니다.
- container 20GB + volume 80GB, HF_HOME=/workspace/huggingface. 생성 화면에서 이미지 크기·디스크 여유·최종 단가를 확인합니다. 제안은 GPU $0.59/h, 총 세전 $2, Pod 생성부터 최대 2시간입니다. 더 비싸거나 공간이 부족하면 생성하지 않고 조건을 다시 검토합니다.

출처: [EXAONE](https://huggingface.co/LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct/tree/553ea250b9a5317231459279d5847d6cf955b9aa), [KURE](https://huggingface.co/nlpai-lab/KURE-v1/tree/8b418a58414668e75532ed045c22d9ca018ae2b2), [vLLM 0.31.0 registry](https://github.com/vllm-project/vllm/blob/v0.31.0/vllm/model_executor/models/registry.py), [서버 인자](https://docs.vllm.ai/en/v0.31.0/cli/serve/), [요금](https://www.runpod.io/pricing).

실행 준비 중 C드라이브 여유 공간 부족을 확인했습니다. 로컬 실행 전 `HF_HOME`을 Git에서 제외된 `ROOT/db/hf-cache`로 설정합니다. 모델 파일은 safetensors 및 토크나이저/설정만 받아 중복 PyTorch 가중치 다운로드를 피합니다. 원본 코퍼스 캐시 위치는 바꾸지 않습니다.


In [8]:
# 공개 설정만 담습니다. 아래 명령은 문자열이며 실행하지 않습니다.
RUNPOD_IMAGE = "vllm/vllm-openai@sha256:c1c9f6fd5c109ba7f0546a59f5b2f15fb87f64c77782e90a27b648b42a8e67c3"
GENERATION_REVISION = "553ea250b9a5317231459279d5847d6cf955b9aa"
EMBEDDING_REVISION = "8b418a58414668e75532ed045c22d9ca018ae2b2"
# 이미지 기본 entrypoint에 넘기는 인자입니다. API 키는 VLLM_API_KEY 환경변수로 주입합니다.
SERVER_ARGS = [BASELINE_SETTINGS["LLM_MODEL"], "--revision", GENERATION_REVISION,
               "--tokenizer-revision", GENERATION_REVISION, "--trust-remote-code", "--dtype", "bfloat16",
               "--max-model-len", "32768", "--max-num-seqs", "1",
               "--gpu-memory-utilization", "0.85", "--generation-config", "vllm",
               "--host", "0.0.0.0", "--port", "8000"]
assert re.fullmatch(r"[0-9a-f]{40}", GENERATION_REVISION)
assert re.fullmatch(r"[0-9a-f]{40}", EMBEDDING_REVISION)
assert "@sha256:" in RUNPOD_IMAGE and "latest" not in RUNPOD_IMAGE
print("고정 이미지:", RUNPOD_IMAGE)
print("서버 인자:", " ".join(SERVER_ARGS))


고정 이미지: vllm/vllm-openai@sha256:c1c9f6fd5c109ba7f0546a59f5b2f15fb87f64c77782e90a27b648b42a8e67c3
서버 인자: LGAI-EXAONE/EXAONE-3.5-7.8B-Instruct --revision 553ea250b9a5317231459279d5847d6cf955b9aa --tokenizer-revision 553ea250b9a5317231459279d5847d6cf955b9aa --trust-remote-code --dtype bfloat16 --max-model-len 32768 --max-num-seqs 1 --gpu-memory-utilization 0.85 --generation-config vllm --host 0.0.0.0 --port 8000


### 노트북이 멈춰도 Pod 중지를 요청하는 별도 프로세스
노트북 timeout은 과금 종료 기능이 아닙니다. 승인 후 Pod 생성 직전에 기록한 UTC 시각 + 2시간을 `RUNPOD_STOP_AT`(Unix 초)에 넣고, 로컬 별도 프로세스에서 아래 함수를 실행합니다. 시계를 다시 시작하거나 자동 연장하지 않습니다.

별도 터미널에서 저장소의 Python으로 실행할 명령:
```text
python -c "import os; from harness.before import stop_runpod_at; print(stop_runpod_at(os.environ['RUNPOD_POD_ID'], float(os.environ['RUNPOD_STOP_AT'])))"
```
`RUNPOD_API_KEY`는 로컬 환경변수로만 전달합니다. vLLM 접근용 `VLLM_API_KEY`와 별개입니다. 실제 호출은 승인 후입니다. Windows에서 도우미를 자동 실행할 때는 숨김 창을 사용합니다.

프로세스가 유지되려면 로컬 PC의 절전·종료를 피해야 합니다. 통신 실패 시 3회까지만 재시도하며 실패하면 콘솔에서 즉시 중지합니다. 이 장치는 엄격한 달러 결제 한도를 보장하지 않습니다. `desiredStatus=EXITED`는 중지 요청 상태이므로 실제 GPU 종료와 청구 내역도 콘솔에서 확인합니다. 결과는 로컬에 저장하고, Pod 파일이 필요하면 백업 확인 후 삭제합니다. 중지 후 volume 저장료는 계속됩니다.

[RunPod 중지 API](https://docs.runpod.io/api-reference/pods/POST/pods/podId/stop) · [상태 조회](https://docs.runpod.io/api-reference/pods/GET/pods/podId) · [저장소 수명](https://docs.runpod.io/pods/storage/types)


In [9]:
def stop_runpod_at(pod_id, stop_at):
    """지정한 Unix 시각에 해당 Pod만 중지 요청하고 API의 기대 상태를 반환합니다."""
    import os
    import requests
    if not re.fullmatch(r"[A-Za-z0-9_-]+", pod_id):
        raise ValueError("invalid_pod_id")
    if not 0 < stop_at < float("inf"):
        raise ValueError("invalid_deadline")
    headers = {"Authorization": "Bearer " + os.environ["RUNPOD_API_KEY"]}
    url = "https://rest.runpod.io/v1/pods/" + pod_id
    # 노트북 커널과 독립된 프로세스에서 대기하도록 함수 전체를 분리합니다.
    while time.time() < stop_at:
        time.sleep(min(30, max(0, stop_at - time.time())))
    for attempt in range(3):
        try:
            response = requests.post(url + "/stop", headers=headers, timeout=15)
            response.raise_for_status()
            response = requests.get(url, headers=headers, timeout=15)
            response.raise_for_status()
            if response.json().get("desiredStatus") == "EXITED":
                return "stop_requested"
        except (requests.RequestException, ValueError):
            pass  # 응답 본문·예외 객체에는 비밀정보가 있을 수 있어 출력하지 않습니다.
        if attempt < 2:
            time.sleep(5)
    raise RuntimeError("Pod 중지 확인 실패: RunPod 콘솔에서 즉시 확인하세요")


In [10]:
# 실제 API·대기 없이 성공, 지연, 통신 실패, 아직 RUNNING인 응답을 검증합니다.
from unittest.mock import Mock, patch
import requests
fake = Mock()
fake.json.return_value = {"desiredStatus": "EXITED"}
with patch.dict(os.environ, {"RUNPOD_API_KEY": "fixture-only"}), \
     patch("requests.post", return_value=fake) as post, \
     patch("requests.get", return_value=fake), \
     patch("time.time", side_effect=[100, 100, 120]), patch("time.sleep") as sleep:
    assert stop_runpod_at("fixture-pod", 120) == "stop_requested"
    sleep.assert_called_once_with(20)
    assert post.call_args.args[0] == "https://rest.runpod.io/v1/pods/fixture-pod/stop"
for failure in (False, True):
    fake.json.return_value = {"desiredStatus": "RUNNING"}
    with patch.dict(os.environ, {"RUNPOD_API_KEY": "fixture-only"}), \
         patch("requests.post", side_effect=requests.Timeout() if failure else None, return_value=fake) as post, \
         patch("requests.get", return_value=fake), patch("time.sleep"):
        try:
            stop_runpod_at("fixture-pod", 1)
        except RuntimeError:
            assert post.call_count == 3
        else:
            raise AssertionError("중지되지 않은 응답을 성공으로 처리함")
print("중지 요청 모의 검증 통과: 실제 Pod/과금 없음")


중지 요청 모의 검증 통과: 실제 Pod/과금 없음


In [11]:
RUN_LIVE = False
# 승인 후 실제 환경 정보를 채웁니다. 비밀키와 엔드포인트는 환경변수로만 전달합니다.
runtime = {"generation_revision": GENERATION_REVISION, "embedding_revision": EMBEDDING_REVISION,
           "quantization": "none (BF16)", "gpu": "A40 48GB x1", "server": RUNPOD_IMAGE,
           "embedding_device": "cpu", "server_args": SERVER_ARGS,
           "index_batch_size": 32, "index_order": "text_length_ascending",
           "execution_packages": execution_packages, "execution_platform": sys.platform,
           "budget_limit": "USD 2 before tax", "time_limit_minutes": 120,
           "stop_at_unix": None, "stop_monitor_started": False, "approved": False}

if RUN_LIVE:
    assert runtime["approved"] and all(v is not None for v in runtime.values()), "환경·비용 승인 필요"
    assert runtime["stop_monitor_started"], "별도 중지 프로세스를 먼저 시작하세요"
    remaining_budget = runtime["stop_at_unix"] - time.time()
    assert 0 < remaining_budget <= 7200, "Pod 생성부터 2시간 이내의 종료 시각 필요"
    deadline = time.monotonic() + remaining_budget
    assert "torch" not in sys.modules, "CPU 설정 적용을 위해 새 커널 필요"
    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    assert not any(name.startswith("app.") for name in sys.modules), "새 커널에서 실행하세요"
    assert sha256(json_bytes({name: sha256((ROOT / name).read_bytes()) for name in app_files})) == manifest["app_files_sha256"]
    assert sha256((SNAPSHOT / "corpus.jsonl").read_bytes()) == manifest["corpus_sha256"]
    assert sha256(SOURCE.read_bytes()) == manifest["source_sha256"]
    assert sha256(EVAL_PATH.read_bytes()) == manifest["eval_sha256"]
    assert os.environ.get("LLM_BASE_URL") and os.environ.get("LLM_API_KEY"), "승인된 엔드포인트 필요"
    for key, value in BASELINE_SETTINGS.items():
        os.environ[key] = str(value)
    os.environ["QDRANT_URL"] = ""
    os.environ["QDRANT_API_KEY"] = ""
    from datetime import datetime, timezone
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    run_dir = SNAPSHOT / "runs" / run_id
    os.environ["QDRANT_PATH"] = str(run_dir / "qdrant")
    os.environ["COLLECTION_LEGAL"] = "before_" + CORPUS_SHA[:12]
    # 승인된 임베딩 revision을 내려받아 해당 경로를 고정합니다. 기본 Run All에서는 실행되지 않습니다.
    from huggingface_hub import snapshot_download
    embedding_path = snapshot_download(BASELINE_SETTINGS["EMBEDDING_MODEL"], revision=runtime["embedding_revision"],
                                       allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt"])
    os.environ["EMBEDDING_MODEL"] = embedding_path
    from app.vectordb import Embedder, QdrantStore
    store = QdrantStore(os.environ["COLLECTION_LEGAL"], Embedder(embedding_path))
    payloads = [service_payload(doc) for doc in documents]
    # 긴 문서에 맞춘 불필요한 패딩을 줄입니다. 내용·모델·검색 설정은 동일합니다.
    payloads.sort(key=lambda item: len(item["text"]))
    for start in range(0, len(payloads), 32):
        if time.monotonic() >= deadline:
            store.client.close()
            raise TimeoutError("index_time_limit")
        batch = payloads[start:start + 32]
        store.add_docs_with_ids([p["text"] for p in batch], [{k: v for k, v in p.items() if k != "text"} for p in batch], wait=True)
        print(f"색인: {min(start + 32, len(payloads))}/{len(payloads)}", flush=True)
    store.client.close()  # 기존 서비스가 같은 로컬 Qdrant를 열 수 있도록 잠금을 해제합니다.
    save_trace(run_dir / "runtime.json", {"runtime": runtime, "manifest": manifest, "run_kind": "reconstructed_before"})
    # 서버가 준비되지 않은 상태를 14개 질문의 성능 실패로 잘못 기록하지 않습니다.
    import requests
    base = os.environ["LLM_BASE_URL"].rstrip("/")
    assert base.endswith("/v1"), "LLM_BASE_URL은 /v1로 끝나야 합니다"
    ready_until = min(deadline, time.monotonic() + 600)
    while True:
        try:
            response = requests.get(base + "/models", headers={"Authorization": "Bearer " + os.environ["LLM_API_KEY"]}, timeout=15)
            if response.status_code == 200:
                served = next(x for x in response.json()["data"] if x["id"] == BASELINE_SETTINGS["LLM_MODEL"])
                assert served.get("max_model_len") == 32768, "승인된 문맥 한도와 다릅니다"
                break
            if response.status_code in (401, 403):
                raise RuntimeError("model_server_auth_failed")
        except requests.RequestException:
            pass
        if time.monotonic() >= ready_until:
            raise TimeoutError("model_server_not_ready; index preserved")
        await asyncio.sleep(10)
    for question in questions:
        remaining = deadline - time.monotonic()
        if remaining <= 0:
            break
        try:
            answer = await asyncio.wait_for(run_current_service(question), timeout=remaining)
        except TimeoutError:
            save_trace(run_dir / (question["id"] + ".json"), {"id": question["id"], "status": "interrupted", "error": "TIME_LIMIT", "usage": "unknown", "run_kind": "reconstructed_before"})
            break
        answer["run_kind"] = "reconstructed_before"
        save_trace(run_dir / (question["id"] + ".json"), answer)
else:
    print("실제 모델·색인 실행 안 함: GPU/비용/시간 승인 대기")

실제 모델·색인 실행 안 함: GPU/비용/시간 승인 대기


### 같은 RunPod에서 전체 실행하기
로컬 PC는 편집·원격 접속·결과 백업만 합니다. vLLM 환경에 프로젝트 의존성을 덮어쓰지 않습니다.

1. 전용 SSH 공개키를 Pod의 기존 PUBLIC_KEY에 추가합니다. 개인키·.env는 Git이나 전송 묶음에 넣지 않습니다.
2. 아래 준비 셀로 Git bundle, 변경 파일, 원문 캐시와 API 복구 입력을 비공개 tar로 묶습니다. 실행 환경은 /workspace/aigo-stage2/project, 자료는 /workspace/aigo-stage2/cache로 분리합니다.
3. SSH로 묶음을 전송하고 SHA-256을 대조한 뒤 /workspace/aigo-stage2 안에서 풉니다. `git clone base.bundle project` 후 overlay의 파일을 project에 복사합니다. 원본 bytes를 보존해 기존 app/eval/corpus 해시와 비교합니다.
4. 별도 Python 3.12 가상환경을 만들고 기존 uv.lock에서 내보낸 제약 파일을 사용합니다. 생성 서버용 torch와 분리하여 CPU torch 2.7.1 및 이 실행에 필요한 패키지만 설치합니다. 실제 버전은 runtime에 기록합니다.
5. AIGO_CACHE_DIR=/workspace/aigo-stage2/cache, HF_HOME=/workspace/huggingface, CUDA_VISIBLE_DEVICES는 빈 값, LLM_BASE_URL=http://127.0.0.1:8000/v1로 설정합니다. 모델 API 키는 Pod의 기존 VLLM_API_KEY에서 프로세스 환경으로만 전달합니다. RunPod 계정 키는 원격 노트북에 전송하지 않습니다.
6. 먼저 Run All을 RUN_LIVE=False로 실행합니다. 이후 확인된 원래 종료 시각과 승인 상태를 실제 실행 사본에 넣어 실행하고, 결과·실행 환경·로그를 로컬에 백업한 후 Pod를 중지합니다.

Linux 실행 패키지는 Windows 준비 manifest와 구분합니다. 입력 해시 비교를 통과했다고 운영 환경을 복원한 것은 아닙니다. 종료 시각과 비용 한도는 사용자 승인 없이 연장하지 않습니다. 후속 승인으로 2026-10-18 17:33:41 KST까지 1시간 연장했으며 총 한도는 $2를 유지했습니다.


실행에서 확인한 준비 사항:
- vLLM 이미지에 SSH 서버가 없었습니다. 웹 터미널에서 openssh-server를 설치하고 비밀번호 로그인을 끈 뒤 기존 공개키로 SSH/SCP 접속했습니다. 터미널 대용량 전송은 실패하여 사용하지 않았습니다.
- 전송 묶음의 로컬·원격 SHA-256은 `97c918a34a2eed14bfd80719e9b724f77915033a6493aacc5b9f0014c9c1501d`로 일치했습니다. 모델 API 키는 컨테이너 환경에서만 읽었습니다.
- 별도 venv에 CPU torch 2.7.1+cpu를 설치했습니다. 나머지는 기존 uv.lock에서 추출한 constraints.txt로 고정했습니다. 설치 대상은 langgraph, langchain-openai, qdrant-client, sentence-transformers, structlog, pydantic-settings, requests, python-dotenv, nbclient, nbformat, ipykernel, fastapi, python-multipart입니다.
- 최초 실제 실행은 fastapi 누락으로 색인 전에 실패했습니다. app.__init__가 API 라우터도 import하므로 필요한 패키지를 추가하고 실패 사본을 보존했습니다.
- CPU 색인은 최초 32개 저장까지 커널 시작 후 약 4분이 걸렸습니다(모듈·모델 로딩 포함). 문자 길이 19~5,051인 문서가 같은 배치에 있어 패딩 비용이 컸습니다. 원문·모델·배치 크기 32를 유지하고 색인 입력만 문자 길이 오름차순으로 정렬했습니다. 문서를 삭제하거나 추가로 자르지 않았습니다. 이전 부분 색인은 보존했습니다.
- OMP/MKL 스레드는 할당 vCPU 수에 맞춰 9로 제한하고 runtime에 기록했습니다. 정렬에 따른 부동소수점 계산·동점 순위의 미세한 차이 가능성이 있으므로 후속 비교에서도 같은 색인 준비 조건을 사용합니다.


In [12]:
PREPARE_REMOTE = False  # 로컬에서 전송 준비할 때만 True. 업로드나 GPU 실행은 하지 않습니다.
if PREPARE_REMOTE:
    import tarfile
    transfer = ROOT / "db" / "runpod-transfer"
    transfer.mkdir(parents=True, exist_ok=True)
    bundle = transfer / "base.bundle"
    subprocess.run(["git", "-c", f"safe.directory={ROOT.as_posix()}", "bundle", "create", str(bundle), "HEAD"], cwd=ROOT, check=True)
    constraints = transfer / "constraints.txt"
    subprocess.run(["uv", "export", "--frozen", "--format", "requirements-txt", "--no-hashes", "--no-emit-project", "--output-file", str(constraints)], cwd=ROOT, check=True, stdout=subprocess.DEVNULL)
    # 명시한 파일만 전송해 .env·키·로컬 실행 로그의 포함을 막습니다.
    selected = app_files + ["pyproject.toml", "uv.lock", "harness/before.py", "harness/notebook_tools.py",
                           "notebooks/redesign/03_metadata_and_ingest.ipynb",
                           "notebooks/redesign/eval/eval_set.json", "notebooks/redesign/data/before_manifest.json"]
    archive = transfer / "before.tar.gz"
    with tarfile.open(archive, "w:gz") as tar:
        tar.add(bundle, arcname="base.bundle")
        tar.add(constraints, arcname="constraints.txt")
        for name in selected:
            tar.add(ROOT / name, arcname="overlay/" + name)
        tar.add(SOURCE, arcname="cache/corpus_v1-seed.jsonl")
        tar.add(CACHE / "manifest_v1-seed.json", arcname="cache/manifest_v1-seed.json")
        for name in lookup_hashes:
            tar.add(CACHE / "api" / name, arcname="cache/api/" + name)
    with tarfile.open(archive) as tar:
        assert all(not Path(m.name).is_absolute() and ".." not in Path(m.name).parts for m in tar.getmembers())
        for name in selected:
            assert sha256(tar.extractfile("overlay/" + name).read()) == sha256((ROOT / name).read_bytes())
        assert sha256(tar.extractfile("cache/corpus_v1-seed.jsonl").read()) == manifest["source_sha256"]
    print("전송 묶음 SHA-256:", sha256(archive.read_bytes()))


## 7. 검증된 함수만 추출
스냅샷·계측 함수만 harness/before.py로 추출합니다. 서비스 코드는 변경하지 않습니다. 재실행 시 기본 live gate가 꺼져 있는지 확인하세요.

In [13]:
from harness.notebook_tools import sync_exports
sync_exports([NOTEBOOK], ROOT, write=True)
assert sync_exports([NOTEBOOK], ROOT) == []
print("03 노트북과 harness/before.py 일치")

03 노트북과 harness/before.py 일치


## 결과·Critical Review·다음 작업
- 문제: 운영 DB 없이 기존 서비스 결과를 재현해야 하며, 시드 캐시에는 날짜 누락과 중복 사건번호가 있습니다.
- 결정: 출처를 포함한 문서 ID, 별도 본문 해시, 원문 bytes 사본으로 고정하고 운영 Before와 구분했습니다. 모의 응답으로 저장 경로만 검증합니다.
- 한계: 캐시 날짜와 최신성은 법률적 재검증이 아닙니다. 모델 revision·이미지 digest와 GPU·예산 제안은 아래 준비 기록에 고정했으며 유료 실행 승인은 대기 중입니다. 실제 응답·usage·비용·검색 점수는 미측정입니다.
- 다음: 코드·자료를 검토한 후 실행 환경과 비용 상한을 승인받습니다. 실제 답변을 얻은 뒤 정성 채점 기준을 보정합니다. Git 반영과 원격 main 병합은 별도 승인입니다.
### 로컬 검증 기록 — 2026-10-17
- 캐시 2,494개: 법령 2,097·판례 342·해석례 55. 정확한 중복 행 0개, 동일 ID의 상이한 본문 충돌 0개. 사건번호 기반 기존 version_key 중복은 1개이나 URL이 달라 두 문서를 모두 보존했습니다.
- 해석례 55개 날짜 누락과 법령 1,711개 현행 별칭 URL을 기록했습니다. 이를 최신성 검증 완료로 표시하지 않았습니다.
- 처음에는 본문 캐시에 법령ID가 없어 빈 ID로 변환할 위험을 확인했습니다. 기존 API 캐시에서 실제 ID 8개를 복구하고 payload 검증을 추가했습니다. 서비스 인용 로직 자체는 변경하지 않았습니다.
- nbformat 구조 검사·새 커널 순차 실행·저장 통과. 입력 순서를 바꿔도 준비 문서가 같고, 원본 bytes와 이전 manifest SHA-1이 일치합니다. 모의 저장·실패·빈 응답·중간 종료 및 실제 서비스 경계의 1회 호출을 검증했습니다.
- `.venv/Scripts/python.exe -m pytest tests/unit/test_evaluation.py tests/unit/test_notebook_tools.py tests/unit/test_sse.py -q -p no:cacheprovider`: 23 passed, 12 subtests passed (11.43s).
- 인증키·주민번호·전화번호 패턴을 새 공개 산출물에 검사했습니다. 제한된 패턴 검사이며 개인정보 전체 검증을 보장하지 않습니다. 원문·API 응답은 Git 밖에 있으며 노트북 출력은 집계·합성 예만 포함합니다.
- manifest는 실제 실행 결과가 아닌 재현 입력 목록입니다. 00/01/03 export 일치 검사와 서비스·기존 하네스·평가셋 무변경 확인을 수행합니다.

### RunPod 준비 검토 — 2026-10-18
- 공개 API·registry에서 모델 SHA 2개와 vLLM 0.31.0 이미지 digest를 조회했습니다. 모델/이미지 파일은 다운로드하지 않았습니다.
- 중지 함수는 모의 API로만 검증합니다. 실제 서버 호환성·GPU 메모리·Pod 중지와 최종 과금은 유료 실행 후 확인할 대상입니다.
- 서비스는 변경하지 않았습니다. 로컬 CPU KURE와 원격 GPU 생성 서버를 분리한 재구성 Before이며 운영 latency 재현이 아닙니다.
- 추가 준비 후 검증: 03 새 커널 실행, 00/01/03 export 일치, 평가·추출·SSE 23 passed / 12 subtests (46.90s), diff 공백 검사 통과. 실제 실행 게이트 False 및 기존 서비스·평가셋 무변경 확인. vLLM 해당 버전의 VLLM_API_KEY 지원도 공식 envs.py에서 확인했습니다.
- 실제 기동 실패 재현: vLLM 0.31.0이 EXAONE custom code 허용 누락으로 ModelConfig ValidationError를 발생시켰습니다. 준비 명령에 --trust-remote-code를 빠뜨린 오류를 수정했습니다. 모델/토크나이저 revision은 유지합니다. 모델 성능 실패가 아니라 서버 준비 오류입니다.
- 실제 로컬 로딩 실패: KURE safetensors를 열 때 Windows OSError 1455(페이징 파일 부족)가 발생했습니다. 색인/질문 수집 전 실패이며 성능 점수는 만들지 않았습니다. 당시 C 여유 0.75 GiB, RAM 31.9 GiB 중 약 8.5 GiB 여유, Windows 가상 메모리 여유 약 2.4 GiB였습니다. 시스템 페이징 설정은 변경하지 않았습니다. 실행 사본은 Git 제외 db/before-execution-20261018.ipynb에 보존합니다.


## 실제 Before 수집 결과 — 2026-10-18
- 문제: 로컬 메모리 부족과 원격 SSH 서버 누락으로 실행 준비가 막혔습니다.
- 해결: 같은 A40 Pod의 별도 CPU 환경에서 KURE·Qdrant·노트북을 실행하고 기존 vLLM 서버로 생성했습니다. CPU 색인은 길이별 배치로 패딩 비용을 줄였습니다. 모델·원문·프롬프트·검색 설정은 유지했습니다.
- 결과: `20261018T073552164357Z` 실행에서 **2,494개 문서 색인과 14개 질문 응답 저장**을 완료했습니다. 응답 저장 성공은 정확성 통과가 아닙니다. 원시 응답·검색 결과·생성 문맥·인용·오류·지연을 보존했습니다. 실측 usage와 비용은 unknown입니다.
- 검증: Linux 새 커널 오프라인 및 실제 순차 실행 통과, 평가셋 ID와 14개 결과 파일 대조, 로컬 복원 SQLite 문서 수 2,494 확인. 실패·중단 실행도 보존했습니다. 신규 설명의 PowerShell 문자 인코딩 문제는 UTF-8로 복구했습니다. 모델 입력·원시 답변은 변경하지 않았습니다.
- 백업: Git 제외 `db/runpod-transfer/before-results.tar.gz`, SHA-256 `faf149564f04b2a01e38cec685aa461160a2bcbd71d6bd1898999a680b11314d`. 원격·로컬 해시가 일치합니다. 결과 사본은 `AIGO_CACHE_DIR/before/<corpus_sha256>/runs/20261018T073552164357Z/`에도 복원했습니다. 백업에는 색인·실행 노트북·패키지 목록이 포함되며 모델 가중치는 포함하지 않습니다.
- 종료: 백업 검증 후 Pod 중지 요청 HTTP 200 및 API 기대 상태 EXITED를 확인했습니다. Pod/volume은 삭제하지 않았습니다. 확정 청구 비용과 저장료는 별도 확인 대상입니다.
- 시간 기록: 사용자가 총 $2 한도를 유지하며 17:33:41 KST까지 60분 연장을 승인했습니다. 실제 종료 경계는 runtime의 stop_at_unix=1792312421.0입니다. 원시 runtime의 time_limit_minutes=120은 최초 승인값이 남은 메타데이터 한계이므로 연장 승인 기록과 함께 읽습니다. 원시 결과는 수정하지 않았습니다.
- Critical Review: 재구성한 환경의 단일 실행이며 운영 환경 성능·통계적 우위를 뜻하지 않습니다. 아래 지연에는 최초 질문의 로딩 영향도 포함됩니다. 검색·생성 품질 및 인용 내용 지지는 아직 채점하지 않았습니다. Q01/Q09/Q12의 조건·판본과 Q10 공고 관련 보류는 유지합니다.
- 다음 작업: 실제 답변으로 사용자와 정성 채점 기준을 보정합니다. commit/push/PR과 merge는 별도 승인 전입니다.

아래 셀은 저장된 결과만 읽습니다. 비공개 결과를 복원하지 않은 환경에서는 요약을 생략합니다. 사람 점수를 자동으로 채우지 않습니다.


In [14]:
from harness.evaluation import percentile

BEFORE_RUN_ID = "20261018T073552164357Z"
before_results = SNAPSHOT / "runs" / BEFORE_RUN_ID
if before_results.exists():
    # 평가셋 ID로 읽어 파일 누락이나 다른 실행의 결과 혼입을 확인합니다.
    before_answers = [json.loads((before_results / f"{q['id']}.json").read_text(encoding="utf-8"))
                      for q in questions]
    assert [r["id"] for r in before_answers] == [q["id"] for q in questions]
    assert all(r["run_kind"] == "reconstructed_before" for r in before_answers)
    elapsed = [r["total_ms"] / 1000 for r in before_answers if r["total_ms"] is not None]
    first_token = [r["ttft_ms"] / 1000 for r in before_answers if r["ttft_ms"] is not None]
    print("응답 상태:", dict(Counter(r["status"] for r in before_answers)))
    print(f"전체 지연(n={len(elapsed)}): P50={percentile(elapsed, 0.5):.2f}s, P95={percentile(elapsed, 0.95):.2f}s")
    print(f"첫 토큰 지연(n={len(first_token)}): P50={percentile(first_token, 0.5):.2f}s")
    print("실제 usage:", dict(Counter(r["usage"] for r in before_answers)))
    print("검색·생성 품질 및 사람 점수: 아직 미채점")
else:
    print("비공개 Before 결과를 복원한 환경에서만 요약합니다.")


응답 상태: {'ok': 14}
전체 지연(n=14): P50=13.14s, P95=22.89s
첫 토큰 지연(n=14): P50=3.25s
실제 usage: {'unknown': 14}
검색·생성 품질 및 사람 점수: 아직 미채점
